# Daily Challenge — Build a Retrieval Augmented Generation (RAG) System
**Week 7 · Day 3**

In this challenge we build a functional **RAG** pipeline with **LangChain** + **Hugging Face** that answers
questions grounded in a real dataset (`databricks/databricks-dolly-15k`).

**Pipeline overview**

1. **Setup** — install the libraries.
2. **Load** the dataset as LangChain `Document`s.
3. **Split** documents into overlapping chunks.
4. **Embed** the chunks with a sentence-transformer.
5. **Index** the embeddings in a **FAISS** vector store.
6. **Prepare** a pre-trained QA model from Hugging Face.
7. **Retrieve + Answer** — connect the retriever to the model.
8. **Test** the system on a query.

> ℹ️ **Note on Step 6–7.** The challenge uses `Intel/dynamic_tinybert`, which is an **extractive** QA model
> (it selects an answer *span* from a given context). LangChain's `HuggingFacePipeline` / `RetrievalQA` only
> accept **generative** LLMs (`text-generation` / `text2text-generation`), so plugging an extractive QA
> pipeline into `RetrievalQA` raises a `ValueError`. We therefore wire retrieval → extractive-QA directly
> (the correct pattern for this model), and add a **bonus** `RetrievalQA` chain using a generative model.


## 1) Set up your environment
Install everything the RAG system needs: LangChain (orchestration), Transformers (models),
sentence-transformers (embeddings), datasets (data), and FAISS (similarity search).

In [1]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu \
    langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Imports (robust across LangChain versions)
from typing import List

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    pipeline,
)

from langchain_core.documents import Document

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

print("Imports OK")

/tmp/ipykernel_4454/2628840713.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Imports OK


## 2) Load the dataset
`databricks/databricks-dolly-15k` has columns `instruction`, `context`, `response`, `category`.
We use the **`context`** column as the retrievable text.

Many rows have an **empty** `context`, so we load the split directly and keep only non-empty contexts
(this is cleaner and faster than indexing thousands of blank documents).

In [3]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

ds = load_dataset(dataset_name, split="train")
print("Raw rows:", len(ds))

# Keep only rows that actually have context text, and wrap them as LangChain Documents.
data: List[Document] = []
for row in ds:
    ctx = (row.get(page_content_column) or "").strip()
    if not ctx:
        continue
    data.append(
        Document(
            page_content=ctx,
            metadata={
                "instruction": row.get("instruction", ""),
                "category": row.get("category", ""),
            },
        )
    )

print("Documents with non-empty context:", len(data))
print("\nExample document:\n", data[0].page_content[:300])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Raw rows: 15011
Documents with non-empty context: 4467

Example document:
 Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major a


## 3) Split the documents
LLMs and embedders have a limited context window. `RecursiveCharacterTextSplitter` breaks long
documents into overlapping chunks so no context is lost at the boundaries.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

docs = text_splitter.split_documents(data)
print("Chunks:", len(docs))
print("\nFirst chunk:\n", docs[0].page_content[:300])

Chunks: 8491

First chunk:
 Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major a


## 4) Embed the text
We turn each chunk into a numerical vector that captures its **semantic meaning**, using the
`all-MiniLM-L6-v2` sentence-transformer. Similar texts end up close together in vector space.

In [5]:
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

# Quick sanity check
query_result = embeddings.embed_query("This is a test document.")
print("Embedding dimension:", len(query_result))
print("First 3 values:", query_result[:3])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
First 3 values: [-0.03833857178688049, 0.12346471846103668, -0.028642943128943443]


## 5) Create a vector store (FAISS)
FAISS indexes the chunk embeddings so we can run **fast similarity search** at query time.

> ⏳ Building the index embeds every chunk — this can take a few minutes depending on how many
> chunks you kept.

In [6]:
db = FAISS.from_documents(
    docs,
    embeddings,
    distance_strategy=DistanceStrategy.COSINE,
)
print("FAISS index built. Vectors:", db.index.ntotal)

FAISS index built. Vectors: 8491


## 6) Prepare the QA model
We load `Intel/dynamic_tinybert`, a fine-tuned **extractive** question-answering model. Given a
`question` and a `context`, it returns the answer **span** found inside that context (plus a
confidence score).

In [8]:
import torch

model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(
    model_name, padding=True, truncation=True, max_length=512
)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# --- Smoke test using manual extraction ---
question = "Where is the Eiffel Tower?"
context = "The Eiffel Tower is a wrought-iron lattice tower located in Paris, France."

# 1. Tokenize the input text pairs together
inputs = tokenizer(question, context, return_tensors="pt")

# 2. Get predictions from the model
with torch.no_grad():
    outputs = model(**inputs)

# 3. Extract the highest scoring start and end positions
start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits)

# 4. Decode the tokens into a readable answer
answer_tokens = inputs.input_ids[0][start_idx : end_idx + 1]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

# Package into a similar dictionary format to match what your exercises expect
demo = {"score": None, "start": int(start_idx), "end": int(end_idx), "answer": answer}
print(demo)

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.bias   | UNEXPECTED |  | 
fit_dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'score': None, 'start': 24, 'end': 26, 'answer': 'paris, france'}


## 7) Build the Retrieval + QA pipeline
The **retriever** finds the most relevant chunks from FAISS; the **QA model** extracts the answer
from those chunks.

Because `Intel/dynamic_tinybert` is *extractive* (not generative), we do **not** use
`RetrievalQA.from_chain_type` here — that chain expects a generative LLM. Instead we retrieve the
top-k chunks, join them into a single context, and let the extractive QA model pull out the answer.
This is the correct way to use this model in a RAG setup.

In [9]:
retriever = db.as_retriever(search_kwargs={"k": 4})

def rag_answer(question: str, k: int = 4):
    """Retrieve top-k chunks from FAISS and extract an answer with the QA model."""
    docs = retriever.invoke(question)[:k]
    context = "\n\n".join(d.page_content for d in docs)

    result = question_answerer(question=question, context=context)
    return {
        "question": question,
        "answer": result["answer"],
        "score": result["score"],
        "source_documents": docs,
    }

print("RAG pipeline ready")

RAG pipeline ready


## 8) Test your RAG system
Ask a question. The system retrieves relevant Dolly contexts and extracts an answer from them.

In [17]:
import torch

def question_answerer(question: str, context: str) -> dict:
    """Custom wrapper to replace the deprecated Hugging Face QA pipeline."""
    # 1. Tokenize the input text pairs together, ensuring truncation to max_length
    inputs = tokenizer(question, context, return_tensors="pt", max_length=512, truncation=True)

    # Explicitly truncate inputs to model's max_position_embeddings if tokenizer failed (e.g., 609 vs 512)
    actual_max_length = model.config.max_position_embeddings # This should be 512 for tinybert
    if inputs['input_ids'].shape[1] > actual_max_length:
        for key in inputs:
            inputs[key] = inputs[key][:, :actual_max_length]

    # 2. Get predictions from the model
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Extract the highest scoring start and end positions
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)

    # 4. Extract raw logit scores for a basic confidence metric
    start_score = float(torch.max(outputs.start_logits))
    end_score = float(torch.max(outputs.end_logits))
    score = (start_score + end_score) / 2  # Simple heuristic for confidence

    # 5. Decode the tokens into a readable answer
    answer_tokens = inputs.input_ids[0][start_idx : end_idx + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

    return {
        "score": score,
        "start": int(start_idx),
        "end": int(end_idx),
        "answer": answer if answer.strip() else "No answer found."
    }

In [18]:
question = "What is cheesemaking?"

result = rag_answer(question=question)

print("Q:", result["question"])
print("A:", result["answer"])
print("Confidence:", round(result["score"], 4))
print("\nRetrieved sources (chunk previews):")
for i, d in enumerate(result["source_documents"], 1):
    print(f"  [{i}] {d.page_content[:120].strip()}...")

Q: What is cheesemaking?
A: to control the spoiling of milk into cheese
Confidence: 6.1991

Retrieved sources (chunk previews):
  [1] The goal of cheese making is to control the spoiling of milk into cheese. The milk is traditionally from a cow, goat, sh...
  [2] Culturing
Cheese is made by bringing milk (possibly pasteurised) in the cheese vat to a temperature required to promote...
  [3] In making Cheddar (or many other hard cheeses) the curd is cut into small cubes and the temperature is raised to approxi...
  [4] Maturing cheese in a cheese cellar
Scalding...


In [19]:
# Try a few more questions
for q in [
    "What is a computer virus?",
    "Who wrote the play Hamlet?",
    "What is machine learning?",
]:
    r = rag_answer(q)
    print(f"Q: {q}\nA: {r['answer']}  (score={r['score']:.3f})\n")

Q: What is a computer virus?
A: a computer worm  (score=4.631)

Q: Who wrote the play Hamlet?
A: shakespeare  (score=4.237)

Q: What is machine learning?
A: a field of inquiry devoted to understanding and building methods that " learn " – that is, methods that leverage data to improve performance on some set of tasks  (score=6.463)



## 🎁 Bonus — a generative `RetrievalQA` chain
To use LangChain's `RetrievalQA` as described in the challenge, we need a **generative** model.
Here we swap in `google/flan-t5-base` (a `text2text-generation` model) so the challenge's
`RetrievalQA.from_chain_type` pattern works end-to-end and produces free-form answers instead of
extracted spans.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

try:
    from langchain_huggingface import HuggingFacePipeline
except ImportError:
    from langchain_community.llms import HuggingFacePipeline

try:
    from langchain.chains import RetrievalQA
except ImportError:
    from langchain_classic.chains import RetrievalQA

gen_id = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_id)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_id)

gen_pipe = pipeline(
    "text2text-generation",
    model=gen_model,
    tokenizer=gen_tokenizer,
    max_length=256,
    do_sample=False,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

out = qa.invoke({"query": "What is cheesemaking?"})
print("Answer:", out["result"])
print("\nSources used:", len(out["source_documents"]))

## ✅ Conclusion
We built a complete **RAG** system:

- Loaded and cleaned `databricks/databricks-dolly-15k` into LangChain `Document`s.
- Chunked the text with `RecursiveCharacterTextSplitter` (1000 / 150 overlap).
- Embedded chunks with `all-MiniLM-L6-v2` and indexed them in **FAISS**.
- Retrieved relevant context and answered questions two ways:
  - **Extractive** QA with `Intel/dynamic_tinybert` (answer spans).
  - **Generative** QA with `flan-t5-base` via LangChain's `RetrievalQA` chain.

**Key takeaway:** match the chain to the model — extractive QA models answer from a supplied context
directly, while `RetrievalQA` expects a generative LLM.
